# Notebook 1: Environment Setup + Model Loading
# สมุดบันทึกที่ 1: การตั้งค่าสภาพแวดล้อม + การโหลดโมเดล

**Master's Thesis: Thai Text-to-Segment System**  
**วิทยานิพนธ์ปริญญาโท: ระบบ Thai Text-to-Segment**

---

This notebook handles:
- สมุดบันทึกนี้จัดการ:
  1. Installing all required dependencies (ติดตั้ง dependencies ที่จำเป็นทั้งหมด)
  2. Verifying GPU and CUDA setup (ตรวจสอบการตั้งค่า GPU และ CUDA)
  3. Loading SAM 3 and InsightFace models (โหลดโมเดล SAM 3 และ InsightFace)
  4. Setting up configuration parameters (ตั้งค่าพารามิเตอร์การกำหนดค่า)

## Section 1: Install Dependencies
## ส่วนที่ 1: ติดตั้ง Dependencies

Install all required packages for the Thai Text-to-Segment system.
ติดตั้งแพ็คเกจที่จำเป็นทั้งหมดสำหรับระบบ Thai Text-to-Segment

In [112]:
# Check Python version
# ตรวจสอบเวอร์ชัน Python
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

Python version: 3.10.12 (main, Nov 20 2023, 15:14:05) [GCC 11.4.0]
Python executable: /root/SEGMENTATION_IVE/.venv/bin/python


In [113]:
# Upgrade pip first
# อัปเกรด pip ก่อน
!pip install --upgrade pip -q

### 1.1 Install PyTorch with CUDA 12.x
### 1.1 ติดตั้ง PyTorch พร้อม CUDA 12.x

In [114]:
# Install PyTorch with CUDA 12.1 support
# ติดตั้ง PyTorch พร้อมรองรับ CUDA 12.1
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

### 1.2 Install SAM 3 (Segment Anything Model 3)
### 1.2 ติดตั้ง SAM 3 (โมเดล Segment Anything 3)

In [115]:
# Install SAM 3 from Hugging Face Transformers
# ติดตั้ง SAM 3 จาก Hugging Face Transformers
# SAM 3 requires transformers >= 4.49.0
!pip install 'transformers>=4.49.0' accelerate -q

In [116]:
# Alternative: Install from source if needed
# ทางเลือก: ติดตั้งจากซอร์สหากจำเป็น
# !pip install git+https://github.com/facebookresearch/sam3.git -q
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 1.3 Install InsightFace
### 1.3 ติดตั้ง InsightFace

In [117]:
# Install InsightFace for face detection and recognition
# ติดตั้ง InsightFace สำหรับการตรวจจับและจดจำใบหน้า
!pip install insightface -q

In [118]:
# Install ONNX Runtime GPU for InsightFace
# ติดตั้ง ONNX Runtime GPU สำหรับ InsightFace
!pip install onnxruntime-gpu -q

### 1.4 Install Additional Dependencies
### 1.4 ติดตั้ง Dependencies เพิ่มเติม

In [119]:
# Install core dependencies
# ติดตั้ง dependencies หลัก
!pip install numpy opencv-python pillow matplotlib -q

In [120]:
# Install transformers and diffusers for text processing
# ติดตั้ง transformers และ diffusers สำหรับการประมวลผลข้อความ
!pip install transformers diffusers accelerate -q

In [121]:
# Install timm for vision models
# ติดตั้ง timm สำหรับโมเดล vision
!pip install timm -q

In [122]:
# Install einops for tensor operations
# ติดตั้ง einops สำหรับการดำเนินการ tensor
!pip install einops -q

In [123]:
# Install scikit-learn for evaluation metrics
# ติดตั้ง scikit-learn สำหรับ metrics การประเมินผล
!pip install scikit-learn scikit-image -q

In [124]:
# Install tqdm for progress bars
# ติดตั้ง tqdm สำหรับแถบความคืบหน้า
!pip install tqdm -q

In [125]:
# Install pyyaml for configuration files
# ติดตั้ง pyyaml สำหรับไฟล์การกำหนดค่า
!pip install pyyaml -q

### 1.5 Install Thai Language Processing Libraries
### 1.5 ติดตั้งไลบรารีการประมวลผลภาษาไทย

In [126]:
# Install pythainlp for Thai text processing
# ติดตั้ง pythainlp สำหรับการประมวลผลข้อความไทย
!pip install pythainlp -q

In [127]:
# Install sentencepiece for tokenization
# ติดตั้ง sentencepiece สำหรับการแบ่งคำ
!pip install sentencepiece -q

## Section 2: Verify GPU and CUDA Setup
## ส่วนที่ 2: ตรวจสอบการตั้งค่า GPU และ CUDA

In [128]:
# Import PyTorch and check CUDA availability
# นำเข้า PyTorch และตรวจสอบความพร้อมใช้งานของ CUDA
import torch
import torchvision

print("=" * 60)
print("GPU and CUDA Verification / การตรวจสอบ GPU และ CUDA")
print("=" * 60)
print(f"\nPyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"\nCUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"\nGPU Details:")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
        print(f"    Compute Capability: {torch.cuda.get_device_properties(i).major}.{torch.cuda.get_device_properties(i).minor}")
else:
    print("\nWARNING: CUDA is not available. The system will use CPU.")
    print("คำเตือน: CUDA ไม่พร้อมใช้งาน ระบบจะใช้ CPU")

GPU and CUDA Verification / การตรวจสอบ GPU และ CUDA

PyTorch version: 2.5.1+cu121
Torchvision version: 0.20.1+cu121

CUDA available: True
CUDA version: 12.1
cuDNN version: 90100
Number of GPUs: 1

GPU Details:
  GPU 0: NVIDIA RTX 6000 Ada Generation
    Memory: 50.87 GB
    Compute Capability: 8.9


In [129]:
# Verify bfloat16 support
# ตรวจสอบการรองรับ bfloat16
print("=" * 60)
print("BFloat16 Support Verification / การตรวจสอบการรองรับ BFloat16")
print("=" * 60)

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    try:
        # Try to create a bfloat16 tensor
        test_tensor = torch.tensor([1.0, 2.0, 3.0], dtype=torch.bfloat16, device=device)
        print(f"\nBFloat16 is supported on {torch.cuda.get_device_name(0)}")
        print(f"BFloat16 รองรับบน {torch.cuda.get_device_name(0)}")
        print(f"Test tensor: {test_tensor}")
        print(f"Tensor dtype: {test_tensor.dtype}")
        BF16_SUPPORTED = True
    except Exception as e:
        print(f"\nBFloat16 is NOT supported: {e}")
        print(f"BFloat16 ไม่รองรับ: {e}")
        BF16_SUPPORTED = False
else:
    print("\nCUDA not available - BFloat16 test skipped")
    BF16_SUPPORTED = False

BFloat16 Support Verification / การตรวจสอบการรองรับ BFloat16

BFloat16 is supported on NVIDIA RTX 6000 Ada Generation
BFloat16 รองรับบน NVIDIA RTX 6000 Ada Generation
Test tensor: tensor([1., 2., 3.], device='cuda:0', dtype=torch.bfloat16)
Tensor dtype: torch.bfloat16


In [130]:
# Test GPU memory allocation
# ทดสอบการจัดสรรหน่วยความจำ GPU
print("=" * 60)
print("GPU Memory Test / ทดสอบหน่วยความจำ GPU")
print("=" * 60)

if torch.cuda.is_available():
    # Get initial memory
    torch.cuda.empty_cache()
    initial_memory = torch.cuda.memory_allocated() / 1e9
    print(f"\nInitial GPU memory allocated: {initial_memory:.2f} GB")
    
    # Create a test tensor
    test_size = 1000
    test_tensor = torch.randn(test_size, test_size, device="cuda")
    
    # Get memory after allocation
    after_alloc = torch.cuda.memory_allocated() / 1e9
    print(f"After allocation ({test_size}x{test_size} tensor): {after_alloc:.2f} GB")
    print(f"Memory used by tensor: {after_alloc - initial_memory:.4f} GB")
    
    # Clean up
    del test_tensor
    torch.cuda.empty_cache()
    final_memory = torch.cuda.memory_allocated() / 1e9
    print(f"After cleanup: {final_memory:.2f} GB")
    print("\nGPU memory test PASSED / การทดสอบหน่วยความจำ GPU ผ่าน")
else:
    print("\nCUDA not available - GPU memory test skipped")

GPU Memory Test / ทดสอบหน่วยความจำ GPU

Initial GPU memory allocated: 3.58 GB
After allocation (1000x1000 tensor): 3.59 GB
Memory used by tensor: 0.0040 GB
After cleanup: 3.58 GB

GPU memory test PASSED / การทดสอบหน่วยความจำ GPU ผ่าน


## Section 3: Import and Verify All Libraries
## ส่วนที่ 3: นำเข้าและตรวจสอบไลบรารีทั้งหมด

In [131]:
# Import all required libraries
# นำเข้าไลบรารีที่จำเป็นทั้งหมด
print("=" * 60)
print("Importing Libraries / กำลังนำเข้าไลบรารี")
print("=" * 60)

# Standard library imports
import os
import sys
import json
import warnings
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

# Third-party imports
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import yaml

# Deep learning imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Thai NLP imports
from pythainlp.tokenize import word_tokenize

print("\nCore libraries imported successfully / นำเข้าไลบรารีหลักสำเร็จ")
print(f"  - NumPy: {np.__version__}")
print(f"  - OpenCV: {cv2.__version__}")
print(f"  - Pillow: {Image.__version__}")
print(f"  - PyThaiNLP: imported")

Importing Libraries / กำลังนำเข้าไลบรารี

Core libraries imported successfully / นำเข้าไลบรารีหลักสำเร็จ
  - NumPy: 2.2.6
  - OpenCV: 4.13.0
  - Pillow: 12.0.0
  - PyThaiNLP: imported


In [132]:
# Import SAM 3 from Hugging Face Transformers
# นำเข้า SAM 3 จาก Hugging Face Transformers
print("\nImporting SAM 3...")
try:
    from transformers import Sam3Processor, Sam3Model
    SAM_VERSION = 3
    print(f"SAM {SAM_VERSION} imported successfully / นำเข้า SAM {SAM_VERSION} สำเร็จ")
except ImportError as e:
    print(f"SAM 3 import failed: {e}")
    SAM_VERSION = None


Importing SAM 3...
SAM 3 imported successfully / นำเข้า SAM 3 สำเร็จ


In [133]:
# Import InsightFace
# นำเข้า InsightFace
print("\nImporting InsightFace...")
try:
    import insightface
    from insightface.app import FaceAnalysis
    print(f"InsightFace imported successfully / นำเข้า InsightFace สำเร็จ")
    print(f"  - InsightFace version: {insightface.__version__ if hasattr(insightface, '__version__') else 'unknown'}")
    INSIGHTFACE_AVAILABLE = True
except ImportError as e:
    print(f"InsightFace import failed: {e}")
    INSIGHTFACE_AVAILABLE = False


Importing InsightFace...
InsightFace imported successfully / นำเข้า InsightFace สำเร็จ
  - InsightFace version: 0.7.3


## Section 4: Load Models
## ส่วนที่ 4: โหลดโมเดล

In [134]:
# Create directories for models
# สร้างไดเรกทอรีสำหรับโมเดล
MODEL_DIR = Path("./models")
SAM_CHECKPOINT_DIR = MODEL_DIR / "sam"
INSIGHTFACE_DIR = MODEL_DIR / "insightface"

SAM_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
INSIGHTFACE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model directories created / สร้างไดเรกทอรีโมเดลแล้ว:")
print(f"  - {SAM_CHECKPOINT_DIR}")
print(f"  - {INSIGHTFACE_DIR}")

Model directories created / สร้างไดเรกทอรีโมเดลแล้ว:
  - models/sam
  - models/insightface


### 4.1 Load SAM Model
### 4.1 โหลดโมเดล SAM

In [135]:
# SAM 3 Model Configuration from Hugging Face
# การกำหนดค่าโมเดล SAM 3 จาก Hugging Face
SAM_CONFIGS = {
    "sam3": {
        "model_id": "facebook/sam3",
        "description": "SAM 3 - Promptable Concept Segmentation"
    }
}

# Select model (SAM 3 has only one variant on Hugging Face)
# เลือกโมเดล (SAM 3 มีเพียงรุ่นเดียวบน Hugging Face)
SELECTED_SAM_MODEL = "sam3"

print(f"Selected SAM model / โมเดล SAM ที่เลือก: {SELECTED_SAM_MODEL}")
print(f"  - Model ID: {SAM_CONFIGS[SELECTED_SAM_MODEL]['model_id']}")
print(f"  - Description: {SAM_CONFIGS[SELECTED_SAM_MODEL]['description']}")

Selected SAM model / โมเดล SAM ที่เลือก: sam3
  - Model ID: facebook/sam3
  - Description: SAM 3 - Promptable Concept Segmentation


In [136]:
# Setup Hugging Face authentication
# ตั้งค่าการยืนยันตัวตนกับ Hugging Face
import os

# Set HF Token for downloading models
# ตั้งค่า HF Token สำหรับดาวน์โหลดโมเดล
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Your HF Token
os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face authentication configured / ตั้งค่าการยืนยันตัวตน Hugging Face แล้ว")
print("SAM 3 will be downloaded from Hugging Face on first use / SAM 3 จะถูกดาวน์โหลดจาก Hugging Face เมื่อใช้งานครั้งแรก")

Hugging Face authentication configured / ตั้งค่าการยืนยันตัวตน Hugging Face แล้ว
SAM 3 will be downloaded from Hugging Face on first use / SAM 3 จะถูกดาวน์โหลดจาก Hugging Face เมื่อใช้งานครั้งแรก


In [137]:
# Load SAM 3 model from Hugging Face
# โหลดโมเดล SAM 3 จาก Hugging Face
print("=" * 60)
print("Loading SAM 3 Model / กำลังโหลดโมเดล SAM 3")
print("=" * 60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device / กำลังใช้อุปกรณ์: {device}")

try:
    if SAM_VERSION == 3:
        # SAM 3 loading from Hugging Face
        sam_config = SAM_CONFIGS[SELECTED_SAM_MODEL]
        model_id = sam_config["model_id"]
        
        print(f"\nLoading SAM 3 from Hugging Face / กำลังโหลด SAM 3 จาก Hugging Face")
        print(f"  - Model ID: {model_id}")
        
        # Load processor and model
        sam_processor = Sam3Processor.from_pretrained(model_id, token=HF_TOKEN)
        sam_model = Sam3Model.from_pretrained(model_id, token=HF_TOKEN).to(device)
        
        print(f"\nSAM 3 model loaded successfully / โหลดโมเดล SAM 3 สำเร็จ!")
        print(f"  - Model: {model_id}")
        print(f"  - Device: {device}")
    else:
        print("SAM 3 not available / SAM 3 ไม่พร้อมใช้งาน")
    
    SAM_LOADED = True
    
except Exception as e:
    print(f"\nError loading SAM 3 model / ข้อผิดพลาดในการโหลดโมเดล SAM 3: {e}")
    SAM_LOADED = False
    sam_model = None
    sam_processor = None

Loading SAM 3 Model / กำลังโหลดโมเดล SAM 3

Using device / กำลังใช้อุปกรณ์: cuda

Loading SAM 3 from Hugging Face / กำลังโหลด SAM 3 จาก Hugging Face
  - Model ID: facebook/sam3


Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 1799.78it/s, Materializing param=vision_encoder.neck.fpn_layers.3.proj2.weight]                       



SAM 3 model loaded successfully / โหลดโมเดล SAM 3 สำเร็จ!
  - Model: facebook/sam3
  - Device: cuda


In [138]:
# Test SAM 3 model with text prompt
# ทดสอบโมเดล SAM 3 ด้วย text prompt
if SAM_LOADED and sam_model is not None and sam_processor is not None:
    print("\nTesting SAM 3 model... / กำลังทดสอบโมเดล SAM 3...")
    
    # Create a dummy image
    from PIL import Image
    import numpy as np
    
    dummy_array = np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)
    dummy_image = Image.fromarray(dummy_array).convert("RGB")
    
    try:
        # Test with text prompt
        inputs = sam_processor(images=dummy_image, text="object", return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = sam_model(**inputs)
        
        # Post-process results
        results = sam_processor.post_process_instance_segmentation(
            outputs,
            threshold=0.5,
            mask_threshold=0.5,
            target_sizes=inputs.get("original_sizes").tolist()
        )[0]
        
        num_masks = len(results["masks"]) if "masks" in results else 0
        
        print(f"  - Input image size: {dummy_image.size}")
        print(f"  - Objects detected: {num_masks}")
        print("\nSAM 3 test PASSED / การทดสอบ SAM 3 ผ่าน!")
        
    except Exception as e:
        print(f"\nSAM 3 test failed / การทดสอบ SAM 3 ล้มเหลว: {e}")
else:
    print("\nSAM 3 model not loaded - skipping test / โมเดล SAM 3 ไม่ได้โหลด - ข้ามการทดสอบ")


Testing SAM 3 model... / กำลังทดสอบโมเดล SAM 3...
  - Input image size: (512, 512)
  - Objects detected: 0

SAM 3 test PASSED / การทดสอบ SAM 3 ผ่าน!


### 4.2 Load InsightFace Model
### 4.2 โหลดโมเดล InsightFace

In [139]:
# InsightFace Configuration
# การกำหนดค่า InsightFace
INSIGHTFACE_CONFIG = {
    "model_name": "buffalo_l",  # Options: buffalo_l, buffalo_m, buffalo_s, buffalo_sc
    "detection_size": (640, 640),
    "detection_threshold": 0.5,
    "ctx_id": 0 if torch.cuda.is_available() else -1  # 0 for GPU, -1 for CPU
}

print(f"InsightFace configuration / การกำหนดค่า InsightFace:")
for key, value in INSIGHTFACE_CONFIG.items():
    print(f"  - {key}: {value}")

InsightFace configuration / การกำหนดค่า InsightFace:
  - model_name: buffalo_l
  - detection_size: (640, 640)
  - detection_threshold: 0.5
  - ctx_id: 0


In [140]:
# Load InsightFace model
# โหลดโมเดล InsightFace
print("=" * 60)
print("Loading InsightFace Model / กำลังโหลดโมเดล InsightFace")
print("=" * 60)

if INSIGHTFACE_AVAILABLE:
    try:
        # Initialize FaceAnalysis app
        face_app = FaceAnalysis(
            name=INSIGHTFACE_CONFIG["model_name"],
            root=str(INSIGHTFACE_DIR),
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'] if torch.cuda.is_available() else ['CPUExecutionProvider']
        )
        
        # Prepare the model with detection size
        face_app.prepare(
            ctx_id=INSIGHTFACE_CONFIG["ctx_id"],
            det_size=INSIGHTFACE_CONFIG["detection_size"],
            det_thresh=INSIGHTFACE_CONFIG["detection_threshold"]
        )
        
        INSIGHTFACE_LOADED = True
        print(f"\nInsightFace model loaded successfully / โหลดโมเดล InsightFace สำเร็จ!")
        print(f"  - Model: {INSIGHTFACE_CONFIG['model_name']}")
        print(f"  - Detection size: {INSIGHTFACE_CONFIG['detection_size']}")
        print(f"  - Device: {'GPU' if INSIGHTFACE_CONFIG['ctx_id'] >= 0 else 'CPU'}")
        
    except Exception as e:
        print(f"\nError loading InsightFace model / ข้อผิดพลาดในการโหลดโมเดล InsightFace: {e}")
        INSIGHTFACE_LOADED = False
        face_app = None
else:
    print("\nInsightFace not available - skipping load / InsightFace ไม่พร้อมใช้งาน - ข้ามการโหลด")
    INSIGHTFACE_LOADED = False
    face_app = None

Loading InsightFace Model / กำลังโหลดโมเดล InsightFace
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: models/insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['

In [141]:
# Test InsightFace model
# ทดสอบโมเดล InsightFace
if INSIGHTFACE_LOADED and face_app is not None:
    print("\nTesting InsightFace model... / กำลังทดสอบโมเดล InsightFace...")
    
    # Create a dummy image with a face-like region
    dummy_face_image = np.zeros((512, 512, 3), dtype=np.uint8)
    # Draw a simple face-like ellipse
    cv2.ellipse(dummy_face_image, (256, 256), (100, 120), 0, 0, 360, (200, 180, 160), -1)
    cv2.circle(dummy_face_image, (220, 230), 15, (50, 50, 50), -1)  # Left eye
    cv2.circle(dummy_face_image, (292, 230), 15, (50, 50, 50), -1)  # Right eye
    cv2.ellipse(dummy_face_image, (256, 290), (40, 25), 0, 0, 180, (100, 50, 50), -1)  # Mouth
    
    try:
        # Detect faces
        faces = face_app.get(dummy_face_image)
        
        print(f"  - Input image shape: {dummy_face_image.shape}")
        print(f"  - Faces detected: {len(faces)}")
        
        if len(faces) > 0:
            for i, face in enumerate(faces):
                print(f"    Face {i+1}:")
                print(f"      - Bounding box: {face.bbox}")
                print(f"      - Detection score: {face.det_score:.4f}")
                print(f"      - Has embedding: {face.embedding is not None}")
        
        print("\nInsightFace test PASSED / การทดสอบ InsightFace ผ่าน!")
        
    except Exception as e:
        print(f"\nInsightFace test failed / การทดสอบ InsightFace ล้มเหลว: {e}")
else:
    print("\nInsightFace model not loaded - skipping test / โมเดล InsightFace ไม่ได้โหลด - ข้ามการทดสอบ")


Testing InsightFace model... / กำลังทดสอบโมเดล InsightFace...
  - Input image shape: (512, 512, 3)
  - Faces detected: 0

InsightFace test PASSED / การทดสอบ InsightFace ผ่าน!


## Section 5: Configuration Setup
## ส่วนที่ 5: การตั้งค่าการกำหนดค่า

In [142]:
# Define project paths
# กำหนดเส้นทางโครงการ
PROJECT_ROOT = Path(".").resolve()

# Directory structure
# โครงสร้างไดเรกทอรี
DIRS = {
    "data": PROJECT_ROOT / "data",
    "data_images": PROJECT_ROOT / "data" / "images",
    "data_annotations": PROJECT_ROOT / "data" / "annotations",
    "data_masks": PROJECT_ROOT / "data" / "masks",
    "models": PROJECT_ROOT / "models",
    "outputs": PROJECT_ROOT / "outputs",
    "outputs_results": PROJECT_ROOT / "outputs" / "results",
    "outputs_visualizations": PROJECT_ROOT / "outputs" / "visualizations",
    "logs": PROJECT_ROOT / "logs",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

# Create all directories
# สร้างไดเรกทอรีทั้งหมด
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

print("Project directories created / สร้างไดเรกทอรีโครงการแล้ว:")
for name, path in DIRS.items():
    print(f"  - {name}: {path}")

Project directories created / สร้างไดเรกทอรีโครงการแล้ว:
  - data: /root/SEGMENTATION_IVE/data
  - data_images: /root/SEGMENTATION_IVE/data/images
  - data_annotations: /root/SEGMENTATION_IVE/data/annotations
  - data_masks: /root/SEGMENTATION_IVE/data/masks
  - models: /root/SEGMENTATION_IVE/models
  - outputs: /root/SEGMENTATION_IVE/outputs
  - outputs_results: /root/SEGMENTATION_IVE/outputs/results
  - outputs_visualizations: /root/SEGMENTATION_IVE/outputs/visualizations
  - logs: /root/SEGMENTATION_IVE/logs
  - checkpoints: /root/SEGMENTATION_IVE/checkpoints


In [143]:
# Model configurations
# การกำหนดค่าโมเดล
MODEL_CONFIG = {
    "sam": {
        "model_name": SELECTED_SAM_MODEL,
        "model_id": SAM_CONFIGS[SELECTED_SAM_MODEL]["model_id"] if SAM_LOADED else None,
        "loaded": SAM_LOADED,
        "version": SAM_VERSION,
        "source": "Hugging Face",
        "device": str(device)
    },
    "insightface": {
        "model_name": INSIGHTFACE_CONFIG["model_name"],
        "root_dir": str(INSIGHTFACE_DIR),
        "loaded": INSIGHTFACE_LOADED,
        "detection_size": INSIGHTFACE_CONFIG["detection_size"],
        "detection_threshold": INSIGHTFACE_CONFIG["detection_threshold"],
        "device": "GPU" if INSIGHTFACE_CONFIG["ctx_id"] >= 0 else "CPU"
    }
}

print("\nModel Configuration / การกำหนดค่าโมเดล:")
print(json.dumps(MODEL_CONFIG, indent=2))


Model Configuration / การกำหนดค่าโมเดล:
{
  "sam": {
    "model_name": "sam3",
    "model_id": "facebook/sam3",
    "loaded": true,
    "version": 3,
    "source": "Hugging Face",
    "device": "cuda"
  },
  "insightface": {
    "model_name": "buffalo_l",
    "root_dir": "models/insightface",
    "loaded": true,
    "detection_size": [
      640,
      640
    ],
    "detection_threshold": 0.5,
    "device": "GPU"
  }
}


In [144]:
# Processing parameters
# พารามิเตอร์การประมวลผล
PROCESSING_CONFIG = {
    "image": {
        "input_size": (1024, 1024),  # Input image size for processing
        "output_size": (512, 512),   # Output mask size
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225]
    },
    "sam": {
        "multimask_output": True,
        "points_per_side": 32,
        "pred_iou_thresh": 0.88,
        "stability_score_thresh": 0.95,
    },
    "insightface": {
        "max_faces": 10,  # Maximum number of faces to detect
        "embeddings_dim": 512,  # Face embedding dimension
    }
}

print("\nProcessing Configuration / การกำหนดค่าการประมวลผล:")
print(json.dumps(PROCESSING_CONFIG, indent=2))


Processing Configuration / การกำหนดค่าการประมวลผล:
{
  "image": {
    "input_size": [
      1024,
      1024
    ],
    "output_size": [
      512,
      512
    ],
    "normalize_mean": [
      0.485,
      0.456,
      0.406
    ],
    "normalize_std": [
      0.229,
      0.224,
      0.225
    ]
  },
  "sam": {
    "multimask_output": true,
    "points_per_side": 32,
    "pred_iou_thresh": 0.88,
    "stability_score_thresh": 0.95
  },
  "insightface": {
    "max_faces": 10,
    "embeddings_dim": 512
  }
}


In [145]:
# Training parameters (for future notebooks)
# พารามิเตอร์การฝึกสอน (สำหรับสมุดบันทึกในอนาคต)
TRAINING_CONFIG = {
    "batch_size": 4 if torch.cuda.is_available() else 1,
    "num_epochs": 50,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 5,
    "save_interval": 5,
    "eval_interval": 1,
    "mixed_precision": BF16_SUPPORTED,  # Use bfloat16 if available
    "gradient_accumulation_steps": 4,
    "max_grad_norm": 1.0,
}

print("\nTraining Configuration / การกำหนดค่าการฝึกสอน:")
print(json.dumps(TRAINING_CONFIG, indent=2))


Training Configuration / การกำหนดค่าการฝึกสอน:
{
  "batch_size": 4,
  "num_epochs": 50,
  "learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "warmup_epochs": 5,
  "save_interval": 5,
  "eval_interval": 1,
  "mixed_precision": true,
  "gradient_accumulation_steps": 4,
  "max_grad_norm": 1.0
}


In [146]:
# Save configuration to YAML file
# บันทึกการกำหนดค่าเป็นไฟล์ YAML
config = {
    "project": {
        "name": "Thai Text-to-Segment System",
        "version": "1.0.0",
        "description": "Master's thesis project for Thai text-guided image segmentation"
    },
    "directories": {k: str(v) for k, v in DIRS.items()},
    "models": MODEL_CONFIG,
    "processing": PROCESSING_CONFIG,
    "training": TRAINING_CONFIG,
    "environment": {
        "python_version": sys.version,
        "pytorch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
        "bfloat16_supported": BF16_SUPPORTED
    }
}

config_path = PROJECT_ROOT / "config.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\nConfiguration saved to / บันทึกการกำหนดค่าไปที่: {config_path}")


Configuration saved to / บันทึกการกำหนดค่าไปที่: /root/SEGMENTATION_IVE/config.yaml


## Section 6: Summary
## ส่วนที่ 6: สรุป

In [147]:
# Print final summary
# แสดงสรุปสุดท้าย
print("=" * 70)
print("SETUP SUMMARY / สรุปการตั้งค่า")
print("=" * 70)

print("\n1. Environment / สภาพแวดล้อม:")
print(f"   - Python: {sys.version.split()[0]}")
print(f"   - PyTorch: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - CUDA Version: {torch.version.cuda}")
    print(f"   - GPU: {torch.cuda.get_device_name(0)}")
print(f"   - BFloat16 Support: {BF16_SUPPORTED}")

print("\n2. Models / โมเดล:")
print(f"   - SAM: {'✓ Loaded' if SAM_LOADED else '✗ Failed'} ({SELECTED_SAM_MODEL})")
print(f"   - InsightFace: {'✓ Loaded' if INSIGHTFACE_LOADED else '✗ Failed'} ({INSIGHTFACE_CONFIG['model_name']})")

print("\n3. Directories / ไดเรกทอรี:")
for name, path in DIRS.items():
    status = "✓" if path.exists() else "✗"
    print(f"   {status} {name}")

print("\n4. Configuration / การกำหนดค่า:")
print(f"   ✓ Saved to: {config_path}")

print("\n" + "=" * 70)
if SAM_LOADED and INSIGHTFACE_LOADED:
    print("✓ SETUP COMPLETE - Ready for next notebook / การตั้งค่าเสร็จสมบูรณ์")
else:
    print("⚠ SETUP INCOMPLETE - Check errors above / การตั้งค่าไม่สมบูรณ์")
print("=" * 70)

SETUP SUMMARY / สรุปการตั้งค่า

1. Environment / สภาพแวดล้อม:
   - Python: 3.10.12
   - PyTorch: 2.5.1+cu121
   - CUDA Available: True
   - CUDA Version: 12.1
   - GPU: NVIDIA RTX 6000 Ada Generation
   - BFloat16 Support: True

2. Models / โมเดล:
   - SAM: ✓ Loaded (sam3)
   - InsightFace: ✓ Loaded (buffalo_l)

3. Directories / ไดเรกทอรี:
   ✓ data
   ✓ data_images
   ✓ data_annotations
   ✓ data_masks
   ✓ models
   ✓ outputs
   ✓ outputs_results
   ✓ outputs_visualizations
   ✓ logs
   ✓ checkpoints

4. Configuration / การกำหนดค่า:
   ✓ Saved to: /root/SEGMENTATION_IVE/config.yaml

✓ SETUP COMPLETE - Ready for next notebook / การตั้งค่าเสร็จสมบูรณ์


In [148]:
# Save summary to file
# บันทึกสรุปลงไฟล์
summary = f"""
Thai Text-to-Segment System - Setup Summary
ระบบ Thai Text-to-Segment - สรุปการตั้งค่า
===========================================

Date / วันที่: {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Environment / สภาพแวดล้อม:
  - Python: {sys.version.split()[0]}
  - PyTorch: {torch.__version__}
  - CUDA Available: {torch.cuda.is_available()}
  - CUDA Version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}
  - GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}
  - BFloat16 Support: {BF16_SUPPORTED}

Models / โมเดล:
  - SAM: {'Loaded' if SAM_LOADED else 'Failed'} ({SELECTED_SAM_MODEL})
  - InsightFace: {'Loaded' if INSIGHTFACE_LOADED else 'Failed'} ({INSIGHTFACE_CONFIG['model_name']})

Status / สถานะ: {'COMPLETE / เสร็จสมบูรณ์' if SAM_LOADED and INSIGHTFACE_LOADED else 'INCOMPLETE / ไม่สมบูรณ์'}
"""

summary_path = DIRS["logs"] / "setup_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(f"Summary saved to / บันทึกสรุปไปที่: {summary_path}")

Summary saved to / บันทึกสรุปไปที่: /root/SEGMENTATION_IVE/logs/setup_summary.txt


---

## Next Steps / ขั้นตอนถัดไป

Proceed to **Notebook 2: Dataset Preparation** / ดำเนินการต่อไปยัง **สมุดบันทึกที่ 2: การเตรียมชุดข้อมูล**

This notebook will:
- สมุดบันทึกนี้จะ:
  - Load and explore the dataset (โหลดและสำรวจชุดข้อมูล)
  - Preprocess images and annotations (ประมวลผลภาพและคำอธิบายประกอบล่วงหน้า)
  - Create data loaders (สร้าง data loaders)
  - Visualize sample data (แสดงภาพตัวอย่างข้อมูล)